# 06 — Finalisation du modèle et test unique sur l'hôpital 2 (Phase 6)

Ce notebook fait deux choses, dans l'ordre :

1. **Entraîne le modèle qui va au test** : la Phase 5 a conclu à une **égalité statistique** entre les 3 techniques anti-shift et la base (écarts de 0,002-0,004 point d'AUC, une seule graine, durées d'entraînement inégales — voir `outputs/rapport/05_robustesse.md`). On retient donc la **base (aucune technique anti-shift)**, choix par défaut qui ne prétend à aucune amélioration non prouvée, avec un seuil déjà proprement établi en Phase 4. Le modèle de la Phase 4 n'a pas survécu à la fin de la session Colab : on le ré-entraîne ici, à l'identique.
2. **Évalue ce modèle, avec son seuil figé avant toute observation de l'hôpital 2**, une seule fois sur l'hôpital 2 — le seul hôpital réellement jamais vu de tout le projet (l'hôpital 1 a servi à tous les réglages : référence, CNN, seuil, variantes).

Voir `PLAN.md` (Phase 6) et `outputs/rapport/06_eval_finale.md`.

In [ ]:
# Colab/Kaggle : clone le depot si besoin et place le repertoire de travail dans notebooks/
# (sans effet en local : le depot est deja la, donc rien n'est clone ni deplace)
import os, subprocess, sys

REPO_URL = "https://github.com/YoucefLgr/Breast_Cancer.git"
EN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
EN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input")

if EN_COLAB or EN_KAGGLE:
    repo_dir = "/content/Breast_Cancer" if EN_COLAB else "/kaggle/working/Breast_Cancer"
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
    os.chdir(os.path.join(repo_dir, "notebooks"))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)

print("Repertoire de travail :", os.getcwd())

In [ ]:
# Paramètres
QUICK = False   # True = sous-ensemble réduit, pour vérifier que tout tourne avant le run complet
VARIANTE_RETENUE = "base"   # Phase 5 : egalite statistique -> choix par defaut, pas un gagnant

# Hôpital 2 : verrou explicite. Ne passer à True que lorsque le modèle et le seuil
# ci-dessous sont définitifs — cette cellule n'est censée s'exécuter qu'UNE SEULE FOIS.
LANCER_TEST = False

import sys, os
sys.path.insert(0, os.path.abspath(".."))
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from src.utils import load_config, set_seed, get_device, detect_environment, evaluer_par_hopital, save_metrics, make_generator
from src.data import charger_sous_echantillon, PatchDataset, construire_transforms, HOPITAL_LABEL
from src.model import construire_modele, geler_backbone, degeler_layers_superieures, compter_parametres
from src.train import entrainer, predire_probas, choisir_seuil

CFG = load_config()
SEED = CFG["seed"]
set_seed(SEED)
DEVICE = get_device()
print("Environnement :", detect_environment(), "| device :", DEVICE)

## 1. Données (train, val_ood — pas encore l'hôpital 2)

In [ ]:
df = charger_sous_echantillon(CFG)
if QUICK:
    morceaux = [g.sample(min(len(g), 60), random_state=SEED)
                for _, g in df.groupby(["split", "center", "label"])]
    df = pd.concat(morceaux, ignore_index=True)
    print("Mode QUICK : sous-ensemble réduit à", len(df), "patchs")

df_train = df[df.split == "train"].reset_index(drop=True)
df_val = df[df.split == "val_ood"].reset_index(drop=True)
print(f"train={len(df_train):,}  val_ood={len(df_val):,}")

t_train, t_eval = construire_transforms(CFG, VARIANTE_RETENUE)
gen = make_generator(SEED)
BS = 32 if QUICK else CFG["modele"]["batch_size"]
NW = 2 if DEVICE == "cuda" else 0
dl_train = DataLoader(PatchDataset(df_train, t_train), batch_size=BS, shuffle=True, generator=gen, num_workers=NW)
dl_val = DataLoader(PatchDataset(df_val, t_eval), batch_size=BS, shuffle=False, num_workers=NW)

## 2. Ré-entraînement du modèle retenu (phase A puis B)

Même protocole que les Phases 4 et 5 : backbone gelé puis `layer3`/`layer4` dégelés, sélection et arrêt anticipé sur `val_ood`.

In [ ]:
modele = construire_modele(CFG["modele"]["architecture"]).to(DEVICE)
geler_backbone(modele)
ep_tete = 1 if QUICK else CFG["modele"]["epochs_tete"]
res_a = entrainer(modele, dl_train, dl_val, df_val.label.values, DEVICE, epochs=ep_tete,
                  lr=CFG["modele"]["lr_tete"], weight_decay=CFG["modele"]["weight_decay"],
                  patience=2, verbeux=True)

degeler_layers_superieures(modele)
ep_ft = 1 if QUICK else CFG["modele"]["epochs_finetune"]
res_b = entrainer(modele, dl_train, dl_val, df_val.label.values, DEVICE, epochs=ep_ft,
                  lr=CFG["modele"]["lr_finetune"], weight_decay=CFG["modele"]["weight_decay"],
                  patience=2, checkpoint_path="../outputs/models/resnet18_final.pt", verbeux=True)
print("Meilleure AUC val_ood (modèle final) :", round(res_b["meilleur_auc_val_ood"], 4))

## 3. Seuil de décision (figé avant de toucher l'hôpital 2)

In [ ]:
proba_val = predire_probas(modele, dl_val, DEVICE)
t_val_05 = evaluer_par_hopital(df_val.label.values, proba_val, df_val.center.values, 0.5, HOPITAL_LABEL)
display(t_val_05)

choix_seuil = choisir_seuil(df_val.label.values, proba_val, CFG["evaluation"]["rappel_cible"])
print(choix_seuil)
t_val_ajuste = evaluer_par_hopital(df_val.label.values, proba_val, df_val.center.values,
                                   choix_seuil["seuil"], HOPITAL_LABEL)
display(t_val_ajuste)

SEUIL_FINAL = choix_seuil["seuil"]
print(f"\nSEUIL FINAL (figé) : {SEUIL_FINAL} — ne sera plus recalculé après ce point.")

## 4. TEST UNIQUE — hôpital 2

**À n'exécuter qu'une seule fois**, quand `SEUIL_FINAL` et `modele` ci-dessus sont définitifs. `LANCER_TEST` doit être explicitement mis à `True` ; `save_metrics(..., overwrite=False)` refuse d'écraser un résultat déjà sauvegardé.

In [ ]:
if not LANCER_TEST:
    print("LANCER_TEST = False : l'hôpital 2 n'est pas chargé. Mettre à True pour le test définitif.")
else:
    df_test = df[df.split == "test"].reset_index(drop=True)
    print(f"Hôpital 2 chargé : {len(df_test):,} patchs — première et unique fois.")
    dl_test = DataLoader(PatchDataset(df_test, t_eval), batch_size=BS, shuffle=False, num_workers=NW)

    proba_test = predire_probas(modele, dl_test, DEVICE)
    t_test_05 = evaluer_par_hopital(df_test.label.values, proba_test, df_test.center.values, 0.5, HOPITAL_LABEL)
    t_test_final = evaluer_par_hopital(df_test.label.values, proba_test, df_test.center.values, SEUIL_FINAL, HOPITAL_LABEL)
    print("-- Hôpital 2, seuil 0,5 --"); display(t_test_05)
    print("-- Hôpital 2, seuil final figé --"); display(t_test_final)

    metrics = {
        "variante_retenue": VARIANTE_RETENUE,
        "seuil_final": SEUIL_FINAL,
        "rappel_cible": CFG["evaluation"]["rappel_cible"],
        "auc_val_ood": res_b["meilleur_auc_val_ood"],
        "metriques_val_ood_seuil_final": t_val_ajuste.to_dict(orient="records"),
        "metriques_hopital2_seuil_0.5": t_test_05.to_dict(orient="records"),
        "metriques_hopital2_seuil_final": t_test_final.to_dict(orient="records"),
    }
    print(save_metrics(metrics, "outputs/metrics/06_eval_finale.json", overwrite=False))

## 5. Lecture

Voir `outputs/rapport/06_eval_finale.md` : comparaison de la performance sur l'hôpital 2 à celle sur `val_ood`, discussion de l'écart (le vrai test de généralisation du projet).